# 02 · Preparación de los datos

**Etapa 3.** Objetivo: construir el conjunto de datos de Argentina listo para analizar, dejando registrado cada paso.

Qué se hace:
1. Cargar los datos originales (que **no se modifican**) y quedarse con Argentina.
2. Pasar los códigos de faltante a `NaN`, guardando antes el motivo.
3. Crear las variables nuevas (grupos de frecuencia de IA, cuartiles de ESCS).
4. Registrar cuántos estudiantes se pierden en cada paso y compararlos con los que quedan.
5. Controlar la calidad y guardar el resultado en `data/processed/`.

No se imputa nada: ni el uso de IA ni el rendimiento.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat

raiz = Path.cwd()
while not (raiz / "README.md").exists():
    if raiz == raiz.parent:
        raise FileNotFoundError("No se encontró README.md: ejecutá el notebook dentro del proyecto.")
    raiz = raiz.parent

ruta_2025 = raiz / "data" / "raw" / "pisa2025" / "extraido" / "CY09_MS_STU_PUF.sav"
carpeta_procesados = raiz / "data" / "processed"
carpeta_tablas = raiz / "reports" / "tables"
carpeta_procesados.mkdir(parents=True, exist_ok=True)
carpeta_tablas.mkdir(parents=True, exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.1.1 | numpy 2.5.4


## 1. Carga de las columnas y filtro de Argentina

Además de lo que se usó en el notebook anterior, se cargan los 10 valores plausibles de cada área y los 80 pesos replicados, que se necesitan para las estimaciones.

In [2]:
ia_4 = ["ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA"]
ia_otras = ["IC170Q10DA", "IC171Q10DA", "AIUSESCH"]

valores_plausibles = []
for area in ["SCIE", "MATH", "READ"]:
    for i in range(1, 11):
        valores_plausibles.append("PV" + str(i) + area)

pesos_replicados = []
for i in range(1, 81):
    pesos_replicados.append("W_FSTURWT" + str(i))

variables = ["CNT", "CNTSCHID", "CNTSTUID", "MALE", "GRADE", "ESCS", "W_FSTUWT"]
variables = variables + ia_4 + ia_otras + valores_plausibles + pesos_replicados

datos, _ = pyreadstat.read_sav(ruta_2025, usecols=variables, user_missing=True)
argentina = datos[datos["CNT"] == "ARG"].copy()
del datos

print("Estudiantes de Argentina:", len(argentina))
print("Columnas:", len(argentina.columns))

Estudiantes de Argentina: 11092
Columnas: 124


## 2. Motivo del faltante y paso a `NaN`

Antes de borrar los códigos se guardan dos datos sobre la pregunta de IA:
- cuántas de las 4 preguntas contestó cada estudiante (`ia_estado`),
- si la pregunta **no se administró** a ese estudiante (código 97), que es distinto de no contestar.

Después, en ESCS y en las variables de IA, los códigos de 95 en adelante pasan a `NaN`.

In [3]:
# 1) Guardamos el motivo del faltante antes de borrar los códigos
contestadas = 0
for pregunta in ia_4:
    contestadas = contestadas + argentina[pregunta].between(1, 5).astype(int)

argentina["ia_cantidad_respondidas"] = contestadas
argentina["ia_estado"] = "parcial"
argentina.loc[contestadas == 4, "ia_estado"] = "respondio_las_4"
argentina.loc[contestadas == 0, "ia_estado"] = "no_respondio_ninguna"

argentina["ia_no_administrado"] = (argentina["ST438Q01DA"] == 97)

print(argentina["ia_estado"].value_counts())
print("IA no administrada:", argentina["ia_no_administrado"].sum())

ia_estado
respondio_las_4         7475
no_respondio_ninguna    3141
parcial                  476
Name: count, dtype: int64
IA no administrada: 526


In [4]:
# 2) Los códigos 95 o más pasan a NaN en ESCS y en las variables de IA
columnas_a_limpiar = ["ESCS"] + ia_4 + ia_otras

for columna in columnas_a_limpiar:
    cantidad = (argentina[columna] >= 95).sum()
    argentina.loc[argentina[columna] >= 95, columna] = np.nan
    print(columna, "-> códigos pasados a NaN:", cantidad)

ESCS -> códigos pasados a NaN: 1284
ST438Q01DA -> códigos pasados a NaN: 3236
ST438Q02DA -> códigos pasados a NaN: 3326
ST438Q03DA -> códigos pasados a NaN: 3412
ST438Q04DA -> códigos pasados a NaN: 3440
IC170Q10DA -> códigos pasados a NaN: 5459
IC171Q10DA -> códigos pasados a NaN: 5833
AIUSESCH -> códigos pasados a NaN: 3364


## 3. Variables nuevas

- **Grupo de frecuencia** de cada actividad de IA: *nunca* (1), *a veces* (2 y 3: una o dos veces al año o al mes) y *seguido* (4 y 5: por semana o todos los días).
- **Promedio de IA**: promedio de las cuatro preguntas, solo para quienes contestaron las cuatro.
- **Cuartiles de ESCS** de Argentina, calculados con el peso final del estudiante (ponderados y relativos a Argentina, no a la OCDE).

In [5]:
# Grupos de frecuencia de cada actividad
for pregunta in ia_4:
    argentina[pregunta + "_grupo"] = pd.cut(
        argentina[pregunta],
        bins=[0, 1, 3, 5],
        labels=["nunca", "a_veces", "seguido"],
    )

# Promedio de las cuatro preguntas (solo si contestó las cuatro)
argentina["ia_promedio"] = argentina[ia_4].mean(axis=1)
argentina.loc[argentina["ia_estado"] != "respondio_las_4", "ia_promedio"] = np.nan

print(argentina["ST438Q04DA_grupo"].value_counts())
print("Estudiantes con promedio de IA:", argentina["ia_promedio"].notna().sum())

ST438Q04DA_grupo
seguido    4019
a_veces    2457
nunca      1176
Name: count, dtype: int64
Estudiantes con promedio de IA: 7475


In [6]:
# Corte ponderado: valor de ESCS por debajo del cual está la proporción pedida de estudiantes
def calcular_corte(valores, pesos, proporcion):
    orden = np.argsort(valores.values)
    valores_ordenados = valores.values[orden]
    pesos_ordenados = pesos.values[orden]
    acumulado = np.cumsum(pesos_ordenados) / pesos_ordenados.sum()
    posicion = np.searchsorted(acumulado, proporcion)
    return valores_ordenados[posicion]

con_escs = argentina[argentina["ESCS"].notna()]
corte_25 = calcular_corte(con_escs["ESCS"], con_escs["W_FSTUWT"], 0.25)
corte_50 = calcular_corte(con_escs["ESCS"], con_escs["W_FSTUWT"], 0.50)
corte_75 = calcular_corte(con_escs["ESCS"], con_escs["W_FSTUWT"], 0.75)
print("Cortes de ESCS (ponderados):", round(corte_25, 3), round(corte_50, 3), round(corte_75, 3))

argentina["cuartil_escs"] = np.nan
argentina.loc[argentina["ESCS"] <= corte_25, "cuartil_escs"] = 1
argentina.loc[(argentina["ESCS"] > corte_25) & (argentina["ESCS"] <= corte_50), "cuartil_escs"] = 2
argentina.loc[(argentina["ESCS"] > corte_50) & (argentina["ESCS"] <= corte_75), "cuartil_escs"] = 3
argentina.loc[argentina["ESCS"] > corte_75, "cuartil_escs"] = 4

print(argentina["cuartil_escs"].value_counts().sort_index())

# Verificamos que con los pesos cada cuartil represente cerca del 25 %
for numero in [1, 2, 3, 4]:
    en_cuartil = argentina[argentina["cuartil_escs"] == numero]
    porcentaje = en_cuartil["W_FSTUWT"].sum() / con_escs["W_FSTUWT"].sum() * 100
    print("Cuartil", numero, "-> % ponderado:", round(porcentaje, 1))

Cortes de ESCS (ponderados): -1.408 -0.708 0.151
cuartil_escs
1.0    2098
2.0    2310
3.0    2433
4.0    2967
Name: count, dtype: int64
Cuartil 1 -> % ponderado: 25.0
Cuartil 2 -> % ponderado: 25.0
Cuartil 3 -> % ponderado: 25.0
Cuartil 4 -> % ponderado: 25.0


## 4. Trazabilidad de exclusiones

Se cuenta cuántos estudiantes quedan según lo que necesita cada hipótesis. Nadie se elimina del archivo: cada análisis usará solo los que tengan las variables que necesita.

In [7]:
total = len(argentina)
con_escs = argentina["ESCS"].notna()
con_ia_4 = argentina["ia_estado"] == "respondio_las_4"
con_covariables = argentina["MALE"].notna() & argentina["GRADE"].notna()

pasos = [
    ("Estudiantes de Argentina", total),
    ("Con ESCS válido (H1)", con_escs.sum()),
    ("Con las 4 preguntas de IA (H3 sin ajustar)", con_ia_4.sum()),
    ("Con ESCS y las 4 preguntas de IA (H2, H3, H4, H5)", (con_escs & con_ia_4).sum()),
    ("Anterior + sexo y grado (H4 ajustado)", (con_escs & con_ia_4 & con_covariables).sum()),
]

tabla_exclusiones = pd.DataFrame(pasos, columns=["muestra", "estudiantes"])
tabla_exclusiones["% del total"] = (tabla_exclusiones["estudiantes"] / total * 100).round(1)
tabla_exclusiones.to_csv(carpeta_tablas / "02_exclusiones.csv", index=False)
print(tabla_exclusiones)

                                             muestra  estudiantes  % del total
0                           Estudiantes de Argentina        11092        100.0
1                               Con ESCS válido (H1)         9808         88.4
2         Con las 4 preguntas de IA (H3 sin ajustar)         7475         67.4
3  Con ESCS y las 4 preguntas de IA (H2, H3, H4, H5)         7191         64.8
4              Anterior + sexo y grado (H4 ajustado)         7191         64.8


**Lectura.** Quedan 7.191 estudiantes (64,8 %) con ESCS y las cuatro preguntas de IA. Hay 1.284 estudiantes sin ESCS válido (código 97) y 3.617 que no contestaron las cuatro preguntas de IA.

## 5. ¿Se parecen los que quedan a los que no?

Si quienes no contestaron las preguntas de IA son distintos (por ejemplo, rinden menos o tienen menor ESCS), los resultados con los que sí contestaron podrían no valer para todos. Se comparan los dos grupos, con y sin pesos.

In [8]:
def promedio_ponderado(columna, pesos):
    validos = columna.notna()
    return np.average(columna[validos], weights=pesos[validos])

contestaron = argentina[argentina["ia_estado"] == "respondio_las_4"]
no_contestaron = argentina[argentina["ia_estado"] != "respondio_las_4"]

variables_a_comparar = ["PV1SCIE", "PV1MATH", "PV1READ", "ESCS", "MALE", "GRADE"]

filas = []
for variable in variables_a_comparar:
    filas.append({
        "variable": variable,
        "contestaron (sin pesos)": round(contestaron[variable].mean(), 3),
        "no contestaron (sin pesos)": round(no_contestaron[variable].mean(), 3),
        "contestaron (con pesos)": round(promedio_ponderado(contestaron[variable], contestaron["W_FSTUWT"]), 3),
        "no contestaron (con pesos)": round(promedio_ponderado(no_contestaron[variable], no_contestaron["W_FSTUWT"]), 3),
    })

comparacion = pd.DataFrame(filas)
comparacion.to_csv(carpeta_tablas / "02_comparacion_respondieron_ia.csv", index=False)
print("Contestaron las 4:", len(contestaron), "| No contestaron las 4:", len(no_contestaron))
print(comparacion)

Contestaron las 4: 7475 | No contestaron las 4: 3617
  variable  contestaron (sin pesos)  no contestaron (sin pesos)  \
0  PV1SCIE                  437.629                     363.254   
1  PV1MATH                  404.904                     339.669   
2  PV1READ                  425.100                     356.404   
3     ESCS                   -0.379                      -0.827   
4     MALE                    0.486                       0.518   
5    GRADE                   -0.084                      -0.274   

   contestaron (con pesos)  no contestaron (con pesos)  
0                  418.488                     355.173  
1                  389.036                     334.734  
2                  412.516                     353.917  
3                   -0.514                      -0.871  
4                    0.485                       0.513  
5                   -0.161                      -0.324  


In [9]:
# Lo mismo separando a quienes no se les administró la pregunta de los que no respondieron
no_administrada = argentina[argentina["ia_no_administrado"]]
sin_respuesta = argentina[(argentina["ia_estado"] != "respondio_las_4") & (~argentina["ia_no_administrado"])]

for nombre, grupo in [("No administrada", no_administrada), ("Sin respuesta / parcial", sin_respuesta)]:
    print(nombre, "| n =", len(grupo),
          "| ciencias:", round(grupo["PV1SCIE"].mean(), 1),
          "| ESCS:", round(grupo["ESCS"].mean(), 3))

No administrada | n = 526 | ciencias: 373.8 | ESCS: nan
Sin respuesta / parcial | n = 3091 | ciencias: 361.5 | ESCS: -0.827


**Lectura.** Los estudiantes que **no** contestaron las cuatro preguntas de IA son distintos de los que sí contestaron: rinden menos en las tres áreas (por ejemplo, en ciencias 355 puntos contra 418 con pesos), tienen menor ESCS (-0,87 contra -0,51) y cursan un grado algo más bajo. La comparación incluye a 526 estudiantes a quienes la pregunta no se administró (tampoco tienen ESCS); la celda siguiente los separa y los dos grupos rinden menos que quienes contestaron (ciencias sin pesos: 374 y 362 contra 438). Esto significa que **los resultados sobre el uso de IA describen a quienes respondieron**, que son un grupo con mejor rendimiento y mayor ESCS que el total. No se puede extender sin cuidado a toda la población. Es una limitación central y una posible hipótesis complementaria (HC1).

## 6. Controles de calidad y guardado

Se comprueban reglas básicas antes de guardar. Si alguna falla, el notebook se detiene con un error.

In [10]:
# 1) No hay estudiantes repetidos
assert argentina["CNTSTUID"].duplicated().sum() == 0

# 2) Solo Argentina
assert (argentina["CNT"] == "ARG").all()

# 3) Las respuestas de IA válidas están entre 1 y 5
for pregunta in ia_4:
    assert argentina[pregunta].dropna().between(1, 5).all()

# 4) ESCS ya no tiene códigos de faltante
assert argentina["ESCS"].dropna().between(-6, 6).all()

# 5) Los 10 valores plausibles de cada área y los 80 pesos replicados están completos
assert argentina[valores_plausibles].notna().all().all()
assert argentina[pesos_replicados].notna().all().all()
assert argentina["W_FSTUWT"].notna().all()

# 6) Cantidad de filas igual a la del archivo original para Argentina
assert len(argentina) == 11092

print("Todos los controles pasaron.")

Todos los controles pasaron.


In [11]:
archivo_salida = carpeta_procesados / "argentina_pisa2025_analitico.csv"
argentina.to_csv(archivo_salida, index=False)

print("Guardado:", archivo_salida.relative_to(raiz))
print("Filas:", len(argentina), "| Columnas:", len(argentina.columns))

Guardado: data\processed\argentina_pisa2025_analitico.csv
Filas: 11092 | Columnas: 133


## 7. Resumen de decisiones de esta etapa

- Los códigos 95 o más de ESCS y de las variables de IA pasaron a `NaN`; antes se guardó el motivo (`ia_estado`, `ia_no_administrado`).
- No se imputó ningún dato. Cada hipótesis usa solo los estudiantes que tienen las variables que necesita (tabla de la sección 4, guardada en `reports/tables/`).
- Cuartiles de ESCS ponderados con el peso final y relativos a Argentina (cortes: -1,408, -0,708 y 0,151).
- Grupos de frecuencia de IA: nunca / a veces / seguido.
- Quienes no respondieron IA son distintos de quienes sí (rinden menos, menor ESCS). Se tendrá en cuenta al interpretar y se estudiará como hipótesis complementaria.
- Conjunto analítico guardado en `data/processed/argentina_pisa2025_analitico.csv` (no se sube al repositorio).